# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles using the `mlcroissant` library.

### Dataset Source
This dataset is described with a Croissant schema available at the FAIR^2 repository.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata fields
ds_metadata = dataset.metadata  # This is an object, not a dict
print(f"Dataset name: {getattr(ds_metadata, 'name', 'N/A')}")
print(f"Description: {getattr(ds_metadata, 'description', 'N/A')}")
print(f"Published: {getattr(ds_metadata, 'date_published', 'N/A')}")

## 2. Data Overview
Let's explore the available record sets, their fields, and get their `@id` values.

Below, we print the available record sets, and for each record set, its fields and columns as referenced by their `@id`.

**Note:** All entity lookups use their `@id` for consistency.

In [ ]:
# List available record sets
print("Record Sets:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"  - Name: {rs.name} | @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for field in rs.fields:
            print(f"      - {field.name} (id: {field.id}, type: {getattr(field, 'data_type', 'N/A')})")
    if hasattr(rs, 'columns'):
        print("    Columns:")
        for col in rs.columns:
            print(f"      - {col.name} (id: {col.id})")

## 3. Data Extraction
Let's load all record sets into Pandas DataFrames and examine the columns for analysis.

**All entity references are by `@id`.**

_Edit the variables below to select which record sets to explore further._

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set into a DataFrame
for rs_id in record_set_ids:
    print(f"Loading records from record set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}:")
    print(df.columns.tolist())
    print(df.head(2), "\n")

## 4. Exploratory Data Analysis (EDA)

For demonstration, let us select a numeric field to filter and normalize, group by a categorical attribute, etc. All column references are by their field `@id` (see above).

Update the variables below to suit your analysis—see the printed record set and field IDs above.

In [ ]:
# Example: Pick a record set and key fields by their @id
# You may need to change these if the IDs do not match your case (see previous cell for correct values)
record_set_id = record_set_ids[0]  # First record set
df = dataframes[record_set_id]

# Pick IDs for a numeric field (e.g., for protein abundance), and group/categorical field (e.g., accession or condition)
all_columns = df.columns.tolist()
numeric_field_id = None
group_field_id = None

print(f"All columns in {record_set_id}:\n{all_columns}")
for c in all_columns:
    # Heuristics: look for common quantity columns
    if numeric_field_id is None and ('abundance' in c or 'molecular_weight' in c or 'MW' in c or 'count' in c):
        numeric_field_id = c
    if group_field_id is None and (c.lower() == 'accession' or c.lower() == 'accession_id' or 'sample' in c):
        group_field_id = c

print(f"\nSelected numeric_field_id: {numeric_field_id}")
print(f"Selected group_field_id: {group_field_id}")

if numeric_field_id is None:
    print("No numeric field detected for EDA.")
else:
    # Remove non-numeric/non-finite
    df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]
    df[numeric_field_id] = df[numeric_field_id].astype(float)

    # Example: filter
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - mean
    ) / std if std else filtered_df[numeric_field_id]
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:\n", grouped.head())

## 5. Visualization
Visualize data distributions such as histograms of numeric fields, or the top values by abundance, grouped by attributes. Edit field IDs as needed!

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Show top 10 values
    top_entries = df.sort_values(numeric_field_id, ascending=False).head(10)
    if group_field_id and group_field_id in top_entries.columns:
        plt.figure(figsize=(10,4))
        plt.bar(top_entries[group_field_id], top_entries[numeric_field_id])
        plt.xticks(rotation=90)
        plt.title(f"Top 10 {group_field_id} by {numeric_field_id}")
        plt.show()

## 6. Conclusion

We demonstrated loading and interactive exploration of a FAIR^2 mass spectrometry dataset using `mlcroissant`. Using only `@id` references, we matched record sets, fields, and columns for robust, reproducible data extraction and analysis workflows. For real research, select attributes of interest based on your task. 

For more details, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the Croissant schema for this dataset.